%md
# 05. Gold – Master Data Integration

## Objetivo

Realizar la integración, normalización y conciliación de los datos maestros de sensores y vehículos con los registros validados de la capa Silver.

Este proceso busca garantizar la consistencia de los identificadores y preparar los datos descriptivos necesarios para la construcción de las dimensiones del modelo dimensional en Gold.

## Alcance del proceso

- Lectura de los archivos maestros desde Landing.
- Normalización de identificadores y atributos.
- Validación de estructura y calidad de los datos.
- Detección y tratamiento de registros duplicados.
- Conciliación de los maestros contra los registros de Silver.
- Identificación de entidades sin correspondencia y registros pendientes de identificación.
- Preparación de los conjuntos de datos para la carga de las dimensiones.

## Resultado esperado

Disponer de conjuntos de datos maestros depurados y conciliados, listos para alimentar las dimensiones de sensores y vehículos en la capa Gold.

## Definición del contrato de datos 

Antes de transformar los maestros, establecemos qué columnas esperamos recibir desde los archivos de entrada

- StructType: define la estructura completa.
- StructField: define una columna, su tipo y si acepta nuloses
- nullable=True: permite valores NULL. False: no los permite.

In [0]:
%python
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, IntegerType, DoubleType
)


# Esquema maestro sensores 

schema_sensores = StructType([
    StructField("sensor_id", StringType(), False),
    StructField("tipo_sensor", StringType(), False),
    StructField("fabricante", StringType(), False),
    StructField("fecha_instalacion", DateType(), False),
    StructField("firmware_version", StringType(), True)
])


# Esquema maestro de vehículos
schema_vehiculos = StructType([
    StructField("vehiculo_id", StringType(), False),
    StructField("marca", StringType(), False),
    StructField("modelo", StringType(), False),
    StructField("tipo_vehiculo", StringType(), False),
    StructField("color", StringType(), True),
    StructField("anio_fabricacion", IntegerType(), False),
    StructField("placa", StringType(), False),
    StructField("capacidad_bateria_kwh", DoubleType(), True),
    StructField("autonomia_km", DoubleType(), True)
])

In [0]:
%python
#1. Lectura de los archivos con  la data maestre aportada por el negocio

#a Rutas de la información a leer

path_sensores = "/Volumes/ecomotive/landing/archivos/Maestro_Sensores/"
path_vehiculos = "/Volumes/ecomotive/landing/archivos/Maestro_Vehiculos/"

#2 revisión de archivos disponibles 

#display(dbutils.fs.ls(path_sensores))
#display(dbutils.fs.ls(path_vehiculos))


#3 lectura de los archivos para crear el dataFrame

df_maestro_sensores=(
     spark.read
     .option("header",'true')
     .schema(schema_sensores)
     .csv(path_sensores)
     )

df_maestro_vehiculos =(
    spark.read
    .option("header",'true')
    .schema(schema_vehiculos)
    .csv(path_vehiculos)
) 



In [0]:
%python

#Revisión del  Esquema de los FD creados (InferSchema intenta determianr los tipos de datos)
df_maestro_sensores.printSchema()
df_maestro_vehiculos.printSchema()

root
 |-- sensor_id: string (nullable = true)
 |-- tipo_sensor: string (nullable = true)
 |-- fabricante: string (nullable = true)
 |-- fecha_instalacion: date (nullable = true)
 |-- firmware_version: string (nullable = true)

root
 |-- vehiculo_id: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- color: string (nullable = true)
 |-- anio_fabricacion: integer (nullable = true)
 |-- placa: string (nullable = true)
 |-- capacidad_bateria_kwh: double (nullable = true)
 |-- autonomia_km: double (nullable = true)



In [0]:
%python
# Reconocimiento de columnas
print("maestro sesores")
display(df_maestro_sensores.limit(5))


print("\n Maestro vehiculos")
display(df_maestro_vehiculos.limit(5))

maestro sesores


sensor_id,tipo_sensor,fabricante,fecha_instalacion,firmware_version
S-1001,Temperatura,Bosch,2025-01-01,v1.0
S-1002,Multisensor,Honeywell,2025-02-02,v2.1
S-1003,Temperatura,Siemens,2025-03-03,v3.2
S-1004,Temperatura,Bosch,2025-04-04,v1.3
S-1006,Multisensor,Honeywell,2025-05-05,v2.4



 Maestro vehiculos


vehiculo_id,marca,modelo,tipo_vehiculo,color,anio_fabricacion,placa,capacidad_bateria_kwh,autonomia_km
T-101,Volvo,FE Electric,Camion electrico,Blanco,2023,EV001,300.0,250.0
T-102,Mercedes-Benz,eActros,Camion electrico,Azul,2024,EV002,400.0,300.0
T-103,Scania,45R,Camion electrico,Gris,2025,EV003,500.0,350.0
T-104,Volvo,FE Electric,Camion electrico,Blanco,2023,EV004,300.0,250.0
T-106,Mercedes-Benz,eActros,Camion electrico,Azul,2024,EV005,400.0,300.0


In [0]:
%python 

# Validación de nombres de columnas recibidas vs esperadas

#.columns: obtiene los nombres de las columnas del DataFrame.
#.fieldNames(): obtiene los nombres definidos en nuestro esquema.
#.set(): convierte las columnas en conjuntos para poder compararlos.

columnas_sensores= set(df_maestro_sensores.columns)
columnas_vehiculos = set(df_maestro_vehiculos.columns)

esperadas_sensores = set(schema_sensores.fieldNames())
esperadas_vehiculos = set(schema_vehiculos.fieldNames())

print("SENSORES")
print("Faltantes:", esperadas_sensores - columnas_sensores)
print("Adicionales:", columnas_sensores - esperadas_sensores)


print("\nVEHÍCULOS")
print("Faltantes:", esperadas_vehiculos - columnas_vehiculos)
print("Adicionales:", columnas_vehiculos - esperadas_vehiculos)




SENSORES
Faltantes: set()
Adicionales: set()

VEHÍCULOS
Faltantes: set()
Adicionales: set()


## Normalización del dataframe

In [0]:
%python
from pyspark.sql.functions import col, trim, upper


df_sensores_normalizados = df_maestro_sensores.withColumn(
    "sensor_id",
    upper(trim(col("sensor_id")))
)

df_vehiculos_normalizados = df_maestro_vehiculos.withColumn(
    "vehiculo_id",
    upper(trim(col("vehiculo_id")))
)

In [0]:
%python

# Detectar IDs duplicados en los maestros normalizados

duplicados_sensores = (
    df_sensores_normalizados
    .groupBy("sensor_id")
    .count()
    .filter("count > 1")
)

duplicados_vehiculos = (
    df_vehiculos_normalizados
    .groupBy("vehiculo_id")
    .count()
    .filter("count > 1")
)

display(duplicados_sensores)
display(duplicados_vehiculos)


sensor_id,count


vehiculo_id,count


## Eliminación de duplicados

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col



# Sensores
window_sensores = Window.partitionBy("sensor_id").orderBy("sensor_id")

df_sensores_unicos = (

    df_sensores_normalizados
    .withColumn("rn", row_number().over(window_sensores))
    .filter(col("rn")==1)
    .drop("rn")

)


# Vehículos
window_vehiculos = Window.partitionBy("vehiculo_id").orderBy("vehiculo_id")

df_vehiculos_unicos = (
    df_vehiculos_normalizados
    .withColumn("rn", row_number().over(window_vehiculos))
    .filter(col("rn") == 1)
    .drop("rn")
)


Determianción de sensores que están en FACT pero no en maestro

In [0]:
%python
# Sensores en Silver que NO están en el maestro
sensores_sin_maestro = (
    spark.table("ecomotive.silver.validos_iot_sensors")
    .select("sensor_id")
    .distinct()
    .join(
        df_sensores_unicos,
        on="sensor_id",
        how="left_anti"
    )
)

# Sensores en el maestro que NO tienen lecturas en Silver
sensores_sin_lecturas = (
    df_sensores_unicos
    .join(
        spark.table("ecomotive.silver.validos_iot_sensors")
        .select("sensor_id")
        .distinct(),
        on="sensor_id",
        how="left_anti"
    )
)


# Vehículos en Silver que NO están en el maestro
vehiculos_sin_maestro = (
    spark.table("ecomotive.silver.validos_iot_sensors")
    .select("vehiculo_id")
    .distinct()
    .join(
        df_vehiculos_unicos,
        on="vehiculo_id",
        how="left_anti"
    )
)

# Vehículos en el maestro que NO tienen lecturas en Silver
vehiculos_sin_lecturas = (
    df_vehiculos_unicos
    .join(
        spark.table("ecomotive.silver.validos_iot_sensors")
        .select("vehiculo_id")
        .distinct(),
        on="vehiculo_id",
        how="left_anti"
    )
)

In [0]:
%python
display(sensores_sin_maestro)
display(sensores_sin_lecturas)

display(vehiculos_sin_maestro)
display(vehiculos_sin_lecturas)

sensor_id


sensor_id,tipo_sensor,fabricante,fecha_instalacion,firmware_version
S-301,Multisensor,Honeywell,2025-08-05,v2.1
S-302,Temperatura,Siemens,2025-09-06,v3.2
S-303,Temperatura,Bosch,2025-10-07,v1.3
S-304,Multisensor,Honeywell,2025-11-08,v2.4
S-305,Temperatura,Siemens,2025-12-09,v3.5
S-306,Temperatura,Bosch,2025-01-10,v1.6
S-307,Multisensor,Honeywell,2025-02-11,v2.7
S-308,Temperatura,Siemens,2025-03-12,v3.8
S-309,Temperatura,Bosch,2025-04-13,v1.9
S-310,Multisensor,Honeywell,2025-05-14,v2.0


vehiculo_id


vehiculo_id,marca,modelo,tipo_vehiculo,color,anio_fabricacion,placa,capacidad_bateria_kwh,autonomia_km
TRK-999,Mercedes-Benz,eActros,Camion electrico,Azul,2024,EV032,400.0,300.0


In [0]:
%python
# Comparación de sensores
ids_silver = (
    spark.table("ecomotive.silver.validos_iot_sensors")
    .select("sensor_id")
    .distinct()
)

ids_maestro = df_sensores_unicos.select("sensor_id").distinct()

print("Sensores en Silver que no están en el maestro:")
display(ids_silver.join(ids_maestro, "sensor_id", "left_anti"))

print("Sensores en el maestro que no están en Silver:")
display(ids_maestro.join(ids_silver, "sensor_id", "left_anti"))

Sensores en Silver que no están en el maestro:


sensor_id


Sensores en el maestro que no están en Silver:


sensor_id
S-301
S-302
S-303
S-304
S-305
S-306
S-307
S-308
S-309
S-310


In [0]:
%python
display(dbutils.fs.ls(path_sensores))
display(dbutils.fs.ls(path_vehiculos))

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/landing/archivos/Maestro_Sensores/maestro_sensores_final.csv,maestro_sensores_final.csv,2067,1794176361000


path,name,size,modificationTime
dbfs:/Volumes/ecomotive/landing/archivos/Maestro_Vehiculos/maestro_vehiculos_final.csv,maestro_vehiculos_final.csv,2345,1794176421000


# Alimentación de  Dimensiones 

In [0]:
%python

#vistatemporal previa alimentación 

df_sensores_unicos.createOrReplaceTempView("vista_sensores")
df_vehiculos_unicos.createOrReplaceTempView("vista_vehiculos")

### Alimnetación dim sensores

In [0]:
-- Carga  dim sensores 

MERGE INTO ecomotive.gold.dim_sensores AS target
USING vista_sensores AS source
ON target.sensor_id = source.sensor_id

WHEN NOT MATCHED THEN
    INSERT (
        sensor_id,
        tipo_sensor,
        fabricante,
        fecha_instalacion,
        firmware_version
    )
    VALUES (
        source.sensor_id,
        source.tipo_sensor,
        source.fabricante,
        source.fecha_instalacion,
        source.firmware_version
    );


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### Alimentación dim vehiculos

In [0]:
--carga Dim vehiculos 

MERGE INTO ecomotive.gold.dim_vehiculos AS target
USING vista_vehiculos AS source

ON target.vehiculo_id = source.vehiculo_id

WHEN NOT MATCHED THEN
    INSERT (
        vehiculo_id,
        marca,
        modelo,
        tipo_vehiculo,
        color,
        anio_fabricacion,
        placa,
        capacidad_bateria_kwh,
        autonomia_km
    )
    VALUES (
        source.vehiculo_id,
        source.marca,
        source.modelo,
        source.tipo_vehiculo,
        source.color,
        source.anio_fabricacion,
        source.placa,
        source.capacidad_bateria_kwh,
        source.autonomia_km
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### Alimentación Dim tiempo

In [0]:
SELECT
 MIN(TO_DATE(momento_de_lectura)) AS fecha_minima,
 MAX(TO_DATE(momento_de_lectura)) AS fecha_maxima
FROM ecomotive.silver.validos_iot_sensors

fecha_minima,fecha_maxima
2025-01-12,2026-11-08


In [0]:
-- Generar calendario completo 2025-2035

CREATE OR REPLACE TEMPORARY VIEW calendario AS

WITH fechas AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE('2025-01-01'),
            DATE('2035-12-31'),
            INTERVAL 1 DAY
        )
    ) AS fecha
)

SELECT
    CAST(DATE_FORMAT(fecha, 'yyyyMMdd') AS INT) AS fecha_key,
    fecha,
    DAY(fecha) AS dia,
    DATE_FORMAT(fecha, 'EEEE') AS nombre_dia,
    DAYOFWEEK(fecha) AS dia_semana,
    WEEKOFYEAR(fecha) AS semana_anio,
    MONTH(fecha) AS mes,
    DATE_FORMAT(fecha, 'MMMM') AS nombre_mes,
    QUARTER(fecha) AS trimestre,
    YEAR(fecha) AS anio,
    DAYOFWEEK(fecha) IN (1, 7) AS es_fin_semana

FROM fechas;

In [0]:
-- Carga incremental de dim_fecha

MERGE INTO ecomotive.gold.dim_fecha AS target

USING calendario AS source

ON target.fecha_key = source.fecha_key

WHEN NOT MATCHED THEN
    INSERT (
        fecha_key,
        fecha,
        dia,
        nombre_dia,
        dia_semana,
        semana_anio,
        mes,
        nombre_mes,
        trimestre,
        anio,
        es_fin_semana
    )

    VALUES (
        source.fecha_key,
        source.fecha,
        source.dia,
        source.nombre_dia,
        source.dia_semana,
        source.semana_anio,
        source.mes,
        source.nombre_mes,
        source.trimestre,
        source.anio,
        source.es_fin_semana
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0
